In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
class BlindnessDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, test=False):
        self.annotations = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform
        self.test = test
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations.iloc[idx, 0] + '.png')
        image = Image.open(img_name)
        
        if self.transform:
            image = self.transform(image)
        
        if self.test:
            return image
        else:
            label = int(self.annotations.iloc[idx, 1])
            return image, label

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
test_csv_file = '/kaggle/input/aptos2019-blindness-detection/test.csv'
test_root_dir = '/kaggle/input/aptos2019-blindness-detection/test_images'
test_dataset = BlindnessDataset(test_csv_file, test_root_dir, transform=transform, test=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [5]:

model_paths = {
    'resnet18': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth",
    'efficientnet_b5': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/efficientnet_b5.pth",
    'inception_resnet_v2': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_resnet_v2.pth",
    'inception_v4': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/inception_v4.pth",
    'seresnext50_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext50_32x4d.pth",
    'seresnext101_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext101_32x4d.pth"
}
"""
model_paths = {
    'seresnext101_32x4d': "/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/seresnext101_32x4d.pth"
}
"""

model_names = {
    'resnet18': 'resnet18',
    'efficientnet_b5': 'efficientnet_b5',
    'inception_resnet_v2': 'inception_resnet_v2',
    'inception_v4': 'inception_v4',
    'seresnext50_32x4d': 'seresnext50_32x4d',
    'seresnext101_32x4d': 'seresnext101_32x4d'
}


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_list = []

# Initialize and load weights for each model using timm
for model_key, path in model_paths.items():
    model_name = model_names[model_key]
    model = timm.create_model(model_name, pretrained=False, num_classes=5)
    model.load_state_dict(torch.load(path))
    model.to(device)
    model.eval()
    models_list.append(model)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/aptos_ensamble-models/pytorch/ensamble_v1/1/resnet18(WD_1e-3)_aptos.pth'

In [7]:
validation_scores = {
    'resnet18': 0.887,
    'efficientnet_b5': 0.952,
    'inception_resnet_v2': 0.822,
    'inception_v4': 0.888,
    'seresnext50_32x4d': 0.709,
    'seresnext101_32x4d': 0.951
}

In [8]:
total_score = sum(validation_scores.values())
weights = {k: v / total_score for k, v in validation_scores.items()}

In [9]:
all_outputs = []

with torch.no_grad():
    for images in tqdm(test_loader):
        images = images.to(device)
        outputs = [weights[model_key] * nn.functional.softmax(model(images), dim=1).unsqueeze(0) 
                   for model_key, model in zip(model_paths.keys(), models_list)]
        outputs = torch.cat(outputs)
        weighted_outputs = torch.sum(outputs, dim=0)
        all_outputs.extend(weighted_outputs.cpu().numpy())

all_outputs = np.array(all_outputs)
final_predictions = np.argmax(all_outputs, axis=1)

  0%|          | 0/23 [00:02<?, ?it/s]


RuntimeError: torch.cat(): expected a non-empty list of Tensors

In [10]:
submission_df = pd.DataFrame({
    'id_code': pd.read_csv(test_csv_file)['id_code'],
    'diagnosis': final_predictions
})

submission_df.to_csv('submission.csv', index=False)

NameError: name 'final_predictions' is not defined